In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 49582 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     49582 non-null  str  
 1   sentiment  49582 non-null  str  
dtypes: str(2)
memory usage: 1.1 MB


### Preprocessing


In [7]:
df['review']=df['review'].str.lower()

In [8]:
import re
sample_text = "abc is the text,abc"

new_text = re.sub("abc","xyz",sample_text)

In [9]:
new_text

'xyz is the text,xyz'

In [10]:
def remove_url(text):
    text=re.sub(r"http\S+","",text)
    return text
df['review']=df['review'].apply(remove_url)

In [11]:
def remove_punctuation(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df['review']=df['review'].apply(remove_punctuation)


In [12]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
df['review']=df['review'].apply(remove_html)

In [13]:
# import nltk
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")


In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
text="I like Coding"
tokens=word_tokenize(text)

In [16]:
tokens

['I', 'like', 'Coding']

In [17]:
def remove_stopwards(text):
    tokens = word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text
df['review']=df['review'].apply(remove_stopwards)

In [18]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [19]:
#running -> run
# played -> play
from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token =ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)
df['review']=df['review'].apply(stemming)


In [20]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [21]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])

In [22]:
y=df['sentiment']

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df['review'])

In [24]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3538)	0.05515484681268887
  (0, 2868)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3002)	0.47200539890110405
  (0, 1275)	0.1357698919196183
  (0, 2289)	0.049500237517476293
  (0, 1933)	0.0791260308382
  (0, 3550)	0.0963974330192639
  (0, 1362)	0.06162489377343992
  (0, 1963)	0.061560444992697486
  (0, 219)	0.08588920995304898
  (0, 1620)	0.0738170550485134
  (0, 4369)	0.041994187696759305
  (0, 4171)	0.17799685402440263
  (0, 3693)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3805)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1739)	0.037520883911174724
  (0, 4497)	0.07614066339174266
  (0, 3857)	0.17537900435282314
  (0, 1630)	0.06142445471882175
  (0, 1862)	0.07433134577032253
  (0, 3329)	0.06406818508428483
  (0, 3332)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1542)	0.17584072573791829

In [25]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [26]:
x_train.shape

(39665, 5000)

In [27]:
x_test.shape

(9917, 5000)

In [28]:
x_train=x_train.toarray()
x_test=x_test.toarray()

In [29]:
import torch
import torch
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32),
)

test_set = TensorDataset(
    torch.tensor(x_test, dtype=torch.float32),
    torch.tensor(y_test.values, dtype=torch.float32),
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)


In [30]:
from torch import optim

In [31]:
import torch.nn as nn
import torch.optim as optimizer
class RNN(nn.Module):
    def __init__ (self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        self.fc = nn.Linear(hidden_size,1)
    def forward(self,x):
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out,_ = self.rnn(x,h0)
        out = self.fc(out[:,-1,:])
        return out


In [32]:
input_size=x_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [33]:
epochs = 10

for epoch in range(epochs):
    model.train()
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        outputs=torch.sigmoid(outputs.squeeze())
        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()

    print(f"{epoch+1}/{epochs} and loss = {loss.item()}")

1/10 and loss = 0.12866400182247162
2/10 and loss = 0.21980009973049164
3/10 and loss = 0.2597402036190033
4/10 and loss = 0.23931598663330078
5/10 and loss = 0.17373421788215637
6/10 and loss = 0.13738998770713806
7/10 and loss = 0.2753661274909973
8/10 and loss = 0.31233566999435425
9/10 and loss = 0.2329424023628235
10/10 and loss = 0.16649912297725677


In [34]:
# evaluate

model.eval()
with torch.no_grad():
    correct_vals = 0
    tot_vals =0
    for Xb,yb in test_loader:
        Xb=Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted= (torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals+=yb.size(0)
        correct_vals += (predicted==yb).sum().item()
    print((correct_vals/tot_vals)*100)


85.74165574266411


In [36]:
import pickle

# Save model
torch.save(model.state_dict(), 'rnn_model.pth')

# Save TF-IDF Vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tf, f)

print("✅ Model saved as 'rnn_model.pth'")
print("✅ TF-IDF Vectorizer saved as 'tfidf_vectorizer.pkl'")
print("Ready for Streamlit deployment!")

✅ Model saved as 'rnn_model.pth'
✅ TF-IDF Vectorizer saved as 'tfidf_vectorizer.pkl'
Ready for Streamlit deployment!
